In [ ]:
#import sys
#!{sys.executable} -m pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://www.google.com"}
)

In [ ]:
sample_doc

In [ ]:
type(sample_doc)

In [ ]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

In [ ]:
document = loader.load()

In [ ]:
document

In [ ]:
# # PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [ ]:
# # PDF Data -- For Complex PDF
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

# Ingestion Pipeline

### Documents

In [ ]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyMuPDFLoader, PyPDFLoader

In [ ]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path

            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs: ",num_docs)
    print("total pages: ", len(all_docs))
    return all_docs

In [ ]:
all_pdf_documents = load_all_pdfs()

### Chunks

In [ ]:
# # Chunks
# import sys
# !{sys.executable} -m pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )


    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [ ]:
chunks = split_docs(all_pdf_documents)

In [ ]:
len(chunks)

### Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model..... ", self.model_name)

        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions= ", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape: ", embeddings.shape)
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

### Vector store

In [ ]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vectore store with collection: ", self.collection_name)
        print("docs in collection: ", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of docs does not match num of embeddings")

        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []


        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total documents added in vector store= ", len(documents_content))
        print("docs in collection: ", self.collection.count())
        

In [ ]:
vector_store = VectorStoreManager()

In [ ]:
# data --> documents --> chunks ---> embeddings --> store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

# Rettrieval Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0):
        # query --> embedding

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine simitlarity

        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1- distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "documents": document,
                        "metadata": metadata,
                         "distance": distance,
                        "similarity_score": similarity_score,
                            "rank" : i+1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documents found")

        return retrieved_docs

                

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
rag_retriever.retrieve("What is RAG?")

# Generation

### Groq

In [ ]:
API_KEY_GROQ = "your_groq_api_key"

In [ ]:
# import sys
# !{sys.executable} -m pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model = "qwen/qwen3-32b",
    temperature= 0.1,
    max_tokens = 1024
     
)

In [ ]:
# generate our retrievalaugmented output

def generate_output(query, retriever, llm , top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use give context generate the answer for the query
                Context: {context}
                Query: {query}
                """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt

    return response.content

In [ ]:
answer = generate_output("What is encode-decoder?", rag_retriever, llm)

In [ ]:
print(answer)